In [1]:
import pandas as pd 
import numpy as np 

df = pd.read_csv("housing.csv")

In [2]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
df["income cat"] = pd.cut(df["median_income"],
                         bins= [0.0,1.5,3.0,4.5,6.0,np.inf],
                         labels=[1,2,3,4,5]
                         )

In [4]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income cat
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,5
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,5
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,5
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,4
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,3


In [5]:
from sklearn.model_selection import train_test_split 

train_set, test_set = train_test_split(df,random_state=42,stratify=df["income cat"],test_size=0.2,)

In [6]:
train_set.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income cat
12655,-121.46,38.52,29.0,3873.0,797.0,2237.0,706.0,2.1736,72100.0,INLAND,2
15502,-117.23,33.09,7.0,5320.0,855.0,2015.0,768.0,6.3373,279600.0,NEAR OCEAN,5
2908,-119.04,35.37,44.0,1618.0,310.0,667.0,300.0,2.8750,82700.0,INLAND,2
14053,-117.13,32.75,24.0,1877.0,519.0,898.0,483.0,2.2264,112500.0,NEAR OCEAN,2
20496,-118.70,34.28,27.0,3536.0,646.0,1837.0,580.0,4.4964,238300.0,<1H OCEAN,3


In [7]:
test_set.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income cat
5241,-118.39,34.12,29.0,6447.0,1012.0,2184.0,960.0,8.2816,500001.0,<1H OCEAN,5
17352,-120.42,34.89,24.0,2020.0,307.0,855.0,283.0,5.0099,162500.0,<1H OCEAN,4
3505,-118.45,34.25,36.0,1453.0,270.0,808.0,275.0,4.3839,204600.0,<1H OCEAN,3
7777,-118.10,33.91,35.0,1653.0,325.0,1072.0,301.0,3.2708,159700.0,<1H OCEAN,3
14155,-117.07,32.77,38.0,3779.0,614.0,1495.0,614.0,4.3529,184000.0,NEAR OCEAN,3


In [8]:
#remove the income cat coolumns
for sett in (train_set,test_set):
    sett.drop("income cat",axis =1,inplace =True)

In [9]:
train_set.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
12655,-121.46,38.52,29.0,3873.0,797.0,2237.0,706.0,2.1736,72100.0,INLAND
15502,-117.23,33.09,7.0,5320.0,855.0,2015.0,768.0,6.3373,279600.0,NEAR OCEAN
2908,-119.04,35.37,44.0,1618.0,310.0,667.0,300.0,2.8750,82700.0,INLAND
14053,-117.13,32.75,24.0,1877.0,519.0,898.0,483.0,2.2264,112500.0,NEAR OCEAN
20496,-118.70,34.28,27.0,3536.0,646.0,1837.0,580.0,4.4964,238300.0,<1H OCEAN


In [10]:
df = train_set.copy()

In [11]:
housing_num = df.drop("ocean_proximity",axis=1).columns.tolist()
housing_cat = ["ocean_proximity"]

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
        ("impute",SimpleImputer(strategy="median")),
        ("Scaler",StandardScaler())
])
num_pipeline

,steps,"[('impute', ...), ('Scaler', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [13]:
from sklearn.preprocessing import OneHotEncoder
cat_pipeline = Pipeline([
        ("one hot encoding",OneHotEncoder())
])
cat_pipeline


,steps,"[('one hot encoding', ...)]"
,transform_input,None
,memory,None
,verbose,False
,categories,'auto'
,drop,None
,sparse_output,True
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None


In [14]:
from sklearn.compose import ColumnTransformer

full_pipeline = ColumnTransformer([
        ("num", num_pipeline, housing_num),
        ("cat",cat_pipeline, housing_cat),
])
full_pipeline

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [15]:
housing_prepared = full_pipeline.fit_transform(df)
housing_prepared.shape

(16512, 14)

In [16]:
columns = full_pipeline.get_feature_names_out()

final_df = pd.DataFrame(housing_prepared,columns=columns,index=df.index)
df_label = final_df["num__median_house_value"]
df_feature = final_df.drop("num__median_house_value",axis= 1)

In [17]:
#model Selction 

#linear regarasar 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score

lin_model = LinearRegression()
lin_model.fit(df_feature,df_label)
lin_model.predict(df_feature)

lin_rmses = -cross_val_score(lin_model,df_feature,df_label,scoring="neg_root_mean_squared_error",cv=10)
print(pd.DataFrame(lin_rmses).describe())

               0
count  10.000000
mean    0.598147
std     0.021611
min     0.564559
25%     0.580170
50%     0.599879
75%     0.611056
max     0.630987


In [18]:
#desosion tree regrasior 
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor

dec_model = DecisionTreeRegressor()
dec_model.fit(df_feature,df_label)
dec_model.predict(df_feature)

dec_rmses = -cross_val_score(dec_model,df_feature,df_label,scoring= "neg_root_mean_squared_error",cv=10)
print(pd.DataFrame(dec_rmses).describe())

               0
count  10.000000
mean    0.598913
std     0.017909
min     0.570225
25%     0.584420
50%     0.600096
75%     0.613557
max     0.624242


In [21]:
#random forest regrasor 
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

ran_model = RandomForestRegressor(random_state=42)
ran_model.fit(df_feature,df_label)
ran_model.predict(df_feature)

ran_rmses = -cross_val_score(ran_model,df_feature,df_label,scoring="neg_root_mean_squared_error",cv=10)
print(pd.DataFrame(ran_rmses).describe())

               0
count  10.000000
mean    0.426749
std     0.018777
min     0.397139
25%     0.413133
50%     0.424942
75%     0.438698
max     0.459720
